In [1]:
!pip install "datasets < 2.20.0"
from datasets import load_dataset, Dataset
import kagglehub
import pandas as pd
import os
import copy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 10.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 13.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.2.0
    Uninstalling fsspec-2026.2.0:
      Successfully uninstalled fsspec-2026.2.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.3
    Uninstalling datasets-4.8.3:
      Successfully uninstalled datasets-4.8.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0

# **LOAD DATASET**

In [2]:
gojek_review_path = kagglehub.dataset_download("ucupsedaya/gojek-app-reviews-bahasa-indonesia")
e_commerce_review_path = kagglehub.dataset_download("satyaahb/e-commerce-sampled-reviews-in-bahasa-indonesia")
print(os.listdir(gojek_review_path))
print(os.listdir(e_commerce_review_path))

['GojekAppReviewV4.0.0-V4.9.3_Cleaned.csv']
['Shopee_Sampled_Reviews.csv']


In [3]:
gojek_reviews_df = pd.read_csv(gojek_review_path + "/GojekAppReviewV4.0.0-V4.9.3_Cleaned.csv")
gojek_reviews_df = gojek_reviews_df[:2500]
gojek_reviews_df.head()

,userName,content,score,at,appVersion
0,Yuga Edit,akun gopay saya di blok,1,2022-01-21 10:52:12,4.9.3
1,ff burik,Lambat sekali sekarang ini bosssku apk gojek g...,3,2021-11-30 15:40:38,4.9.3
2,Anisa Suci Rahmayuliani,Kenapa sih dari kemarin sy buka aplikasi gojek...,4,2021-11-29 22:58:12,4.9.3
3,naoki yakuza,Baru download gojek dan hape baru trus ditop u...,1,2022-09-03 15:21:17,4.9.3
4,Trio Sugianto,Mantap,5,2022-01-15 10:05:27,4.9.3


In [4]:
e_commerce_reviews_df = pd.read_csv(e_commerce_review_path + "/Shopee_Sampled_Reviews.csv")
e_commerce_reviews_df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt
0,61ccddf5-2848-47d6-83a7-434e4e613bfa,Andi Gunawan,https://play-lh.googleusercontent.com/a-/ACB-R...,Udah sering belanja trs tapi setiap pengajuan ...,1,1,2.95.47,2022-12-05 13:29:47,"Hi kak, maaf ya buat gak nyaman. terkait kenda...",2022-11-04 16:59:58
1,affdfdc0-0a10-4353-8ba9-52a669f8a1ba,Sari Sari,https://play-lh.googleusercontent.com/a/AGNmyx...,Semenjak di upgrade.. SHOPEE JADI LEMOT,1,0,NaN,2022-12-27 05:36:57,Hi kak maaf atas ketidaknyamannya🙏 Pastiin RAM...,2022-12-27 06:14:13
2,f5a73edb-ae1a-4a1b-93a6-aa2a5fe5217e,Laz Ai,https://play-lh.googleusercontent.com/a/AGNmyx...,Penyelesaian masalah sangat buruk,1,0,NaN,2022-08-15 07:00:00,NaN,NaN
3,0ffb52f7-3611-4fd3-a874-6e3ef6ad4fed,Kuprit Bae,https://play-lh.googleusercontent.com/a/AGNmyx...,Apk engga😇 jls,1,0,2.52.07,2023-03-16 04:05:30,"Hai kak, maaf ya bikin ga nyaman. Kedepannya S...",2023-03-16 05:17:45
4,c726e46a-3343-4db0-8733-30e8b42d9f1c,Evans irdas,https://play-lh.googleusercontent.com/a-/ACB-R...,Lelet stress. Udah update terbaru tetap aja lemot,1,0,2.95.52,2022-12-22 17:34:33,NaN,NaN


In [5]:
transitional_df = pd.read_json("https://raw.githubusercontent.com/fadhilr2/AI_TAHAP_2/refs/heads/main/data_fetching/transitional/datasets/data.jsonl", lines=True)
transitional_df.head()

,content,label
0,"Halo kak, mau tanya nih. Untuk produk yang ini...",1
1,"Permisi, mau nanya harga tas selempang yang la...",1
2,"Kak, kalau untuk sepatu sneakers model A itu, ...",1
3,"Halo, kak. Untuk paket bundling produk skincar...",1
4,"Sore kak, mau tanya harga jam tangan kulit yan...",1


# **DATA PREPROCESSING**

In [6]:
def map_stars_to_label(score):
    if score <= 2:
        return 0
    elif score >= 4:
        return 2
    else:
        return None

gojek_reviews_df["label"] = gojek_reviews_df["score"].apply(map_stars_to_label)
gojek_reviews_df = gojek_reviews_df.dropna(subset=["label"])
gojek_reviews_df["label"] = gojek_reviews_df["label"].astype(int)

e_commerce_reviews_df["label"] = e_commerce_reviews_df["score"].apply(map_stars_to_label)
e_commerce_reviews_df = e_commerce_reviews_df.dropna(subset=["label"])
e_commerce_reviews_df["label"] = e_commerce_reviews_df["label"].astype(int)
print("done")

done


In [7]:
gojek_reviews_df = gojek_reviews_df.drop(columns=["userName",
                                                  "at",
                                                  "appVersion",
                                                  "score"])
gojek_reviews_df.head()

,content,label
0,akun gopay saya di blok,0
2,Kenapa sih dari kemarin sy buka aplikasi gojek...,2
3,Baru download gojek dan hape baru trus ditop u...,0
4,Mantap,2
5,Bagus,2


In [8]:
e_commerce_reviews_df = e_commerce_reviews_df.drop(columns=["reviewId",
                                                            "userName",
                                                            "userImage",
                                                            "thumbsUpCount",
                                                            "reviewCreatedVersion",
                                                            "at",
                                                            "replyContent",
                                                            "repliedAt",
                                                            "score"])
e_commerce_reviews_df.head()

,content,label
0,Udah sering belanja trs tapi setiap pengajuan ...,0
1,Semenjak di upgrade.. SHOPEE JADI LEMOT,0
2,Penyelesaian masalah sangat buruk,0
3,Apk engga😇 jls,0
4,Lelet stress. Udah update terbaru tetap aja lemot,0


In [9]:
df = pd.concat([gojek_reviews_df, e_commerce_reviews_df, transitional_df], ignore_index=True)
df["label"] = df["label"].astype(int)
df.head()

,content,label
0,akun gopay saya di blok,0
1,Kenapa sih dari kemarin sy buka aplikasi gojek...,2
2,Baru download gojek dan hape baru trus ditop u...,0
3,Mantap,2
4,Bagus,2


# **SMsA FINE TUNING**

In [10]:
!pip install transformers torch

In [11]:
!pip install "numpy<2.0"
!pip install evaluate
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoModel, TrainingArguments, Trainer, pipeline
import evaluate
import numpy as np


In [12]:
def tokenize_function(examples):
  return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128 
    )

# INITIAL FINE TUNING

In [13]:
smsa = load_dataset("indonlp/indonlu", "smsa")

model_name = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenized_datasets = smsa.map(tokenize_function, batched=True)

/usr/local/lib/python3.12/dist-packages/datasets/load.py:1491: FutureWarning: The repository for indonlp/indonlu contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/indonlp/indonlu
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating train split:   0%|          | 0/11000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1260 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

config.json: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/11000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1260 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

In [15]:
training_args = TrainingArguments(
    output_dir="./indobert-smsa-results",
    eval_strategy="epoch",          
    save_strategy="epoch",            
    learning_rate=2e-5,               
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,               
    weight_decay=0.01,
    load_best_model_at_end=True,      
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.373029,0.929365
2,0.398260,0.408857,0.936508
3,0.148245,0.524885,0.935714


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1032, training_loss=0.2692460399265437, metrics={'train_runtime': 513.5075, 'train_samples_per_second': 64.264, 'train_steps_per_second': 2.01, 'total_flos': 2170685696256000.0, 'train_loss': 0.2692460399265437, 'epoch': 3.0})

In [16]:
save_path = '/kaggle/working/my_model'
if not os.path.exists(save_path):
    os.makedirs(save_path)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/my_model/tokenizer_config.json',
 '/kaggle/working/my_model/tokenizer.json')

# USING TRAINED MODEL

In [17]:
# !pip install gdown
# import gdown


In [18]:
# file_id = '1hXz7yZ2aK2LmnqyCoaaTvmG1cLtVxsSO'
# url = f'https://drive.google.com/uc?id={file_id}'
# output = 'my_indobert_checkpoint.zip'
# gdown.download(url, output, quiet=False)
# !unzip my_indobert_checkpoint.zip -d /kaggle/working/my_model

In [19]:
import os

In [20]:
path = '/kaggle/working/my_model'
model = AutoModelForSequenceClassification.from_pretrained(path)
tokenizer = AutoTokenizer.from_pretrained(path)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [21]:

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)


texts = [
    "Barangnya bagus banget, sesuai ekspektasi!",
    "Biasa aja sih, nggak ada yang spesial.",
    "Pengiriman lambat, barang sampai dalam keadaan rusak. Kecewa parah."
]

predictions = classifier(texts)

for text, pred in zip(texts, predictions):
    print(f"Text: {text}\nPrediction: {pred}\n")

Text: Barangnya bagus banget, sesuai ekspektasi!
Prediction: {'label': 'LABEL_0', 'score': 0.9955005049705505}

Text: Biasa aja sih, nggak ada yang spesial.
Prediction: {'label': 'LABEL_2', 'score': 0.8594577312469482}

Text: Pengiriman lambat, barang sampai dalam keadaan rusak. Kecewa parah.
Prediction: {'label': 'LABEL_2', 'score': 0.9710670113563538}



# **FINE TUNING ON CUSTOM DATASETS**

In [22]:
print("Label distribution before split:")
print(df["label"].value_counts().sort_index())

Label distribution before split:
label
0    1780
1    2923
2    2589
Name: count, dtype: int64


In [23]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42
)

In [24]:
print(val_df)

                                                content  label
6454  Halo, aku pernah beli jam tangan strap kulit. ...      1
7173  Permisi kak, untuk motif ini, ada pilihan warn...      1
6571  Kak, mau tanya soal obat jerawat. Kemarin beli...      1
658   Saya driver Gojek saya mengeluh orderan yg say...      2
763                              Baikkkkkkk mantap joss      2
...                                                 ...    ...
1471                                 Cepet aja pokoknya      2
3602                            Shopee mantap memuaskan      2
725                                              Mantab      2
2422  Kg jelas ngaleg scroll nya leg, ini udh parah ...      0
4710  Mau tanya harga paket wedding organizer dong. ...      1

[1094 rows x 2 columns]


In [25]:
print(f"\nTrain: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")



Train: 5104 | Val: 1094 | Test: 1094


In [26]:
dataset_train_df = Dataset.from_pandas(train_df)
dataset_val_df = Dataset.from_pandas(val_df)
dataset_test_df = Dataset.from_pandas(test_df)

dataset_train_df = dataset_train_df.rename_column("content", "text")
dataset_val_df = dataset_val_df.rename_column("content", "text")
dataset_test_df = dataset_test_df.rename_column("content", "text")

In [27]:
train_ds = dataset_train_df.map(tokenize_function, batched=True)
val_ds   = dataset_val_df.map(tokenize_function, batched=True)
test_ds  = dataset_test_df.map(tokenize_function, batched=True)

Map:   0%|          | 0/5104 [00:00<?, ? examples/s]

Map:   0%|          | 0/1094 [00:00<?, ? examples/s]

Map:   0%|          | 0/1094 [00:00<?, ? examples/s]

In [28]:
cols = ["input_ids", "attention_mask", "token_type_ids", "label"]
train_ds.set_format(type="torch", columns=cols)
val_ds.set_format(type="torch", columns=cols)
test_ds.set_format(type="torch", columns=cols)

In [29]:
id2label = {0: "Transactional", 1: "Transitional", 2: "Communal"}
label2id = {"Transactional": 0, "Transitional": 1, "Communal": 2}

model = AutoModelForSequenceClassification.from_pretrained(
    path,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [30]:
import torch
from torch.nn import CrossEntropyLoss

label_counts = train_df["label"].value_counts().sort_index()
total = len(train_df)

weights = torch.tensor(
    [total / label_counts[i] for i in range(3)],
    dtype=torch.float
)
weights = weights / weights.sum()  

print("Class weights:", weights)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = weights.to(device)

Class weights: tensor([0.4354, 0.2652, 0.2994])


In [31]:
from torch.nn import CrossEntropyLoss
from sklearn.metrics import classification_report, f1_score

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fn = CrossEntropyLoss(weight=weights)
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    macro_f1 = f1_score(labels, preds, average="macro")

    report = classification_report(
        labels, preds,
        target_names=["Transactional", "Transitional", "Communal"],
        output_dict=True
    )

    return {
        "macro_f1":             macro_f1,
        "f1_transactional":     report["Transactional"]["f1-score"],
        "f1_transitional":      report["Transitional"]["f1-score"],
        "f1_communal":          report["Communal"]["f1-score"],
    }

In [32]:
training_args = TrainingArguments(
    output_dir="./indobert-crm",

    num_train_epochs=5,
    learning_rate=2e-5,            
    weight_decay=0.01,
    warmup_steps=100,              

    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    logging_steps=50,
    report_to="none"               
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1,F1 Transactional,F1 Transitional,F1 Communal
1,1.357232,0.324277,0.872747,0.783582,0.987599,0.847059
2,0.251294,0.267618,0.901597,0.833935,0.993197,0.877660
3,0.196011,0.257412,0.900949,0.828467,0.996587,0.877792
4,0.144741,0.290416,0.903081,0.833028,0.994325,0.881890
5,0.115326,0.328414,0.895106,0.816479,0.994325,0.874515


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=400, training_loss=0.3359331202507019, metrics={'train_runtime': 357.5905, 'train_samples_per_second': 71.367, 'train_steps_per_second': 1.119, 'total_flos': 1678663605104640.0, 'train_loss': 0.3359331202507019, 'epoch': 5.0})

In [33]:
from sklearn.metrics import classification_report, confusion_matrix

results = trainer.predict(test_ds)
preds   = np.argmax(results.predictions, axis=-1)
labels  = results.label_ids

print(classification_report(
    labels, preds,
    target_names=["Transactional", "Transitional", "Communal"]
))

print("Confusion matrix:")
print(confusion_matrix(labels, preds))

               precision    recall  f1-score   support

Transactional       0.81      0.84      0.82       267
 Transitional       1.00      1.00      1.00       439
     Communal       0.89      0.86      0.87       388

     accuracy                           0.91      1094
    macro avg       0.90      0.90      0.90      1094
 weighted avg       0.91      0.91      0.91      1094

Confusion matrix:
[[225   1  41]
 [  0 439   0]
 [ 54   1 333]]


In [34]:
save_path = '/kaggle/working/indobert_trained'
if not os.path.exists(save_path):
    os.makedirs(save_path)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/indobert_trained/tokenizer_config.json',
 '/kaggle/working/indobert_trained/tokenizer.json')

In [35]:

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)


texts = [
    "Barangnya bagus banget, sesuai ekspektasi!",
    "Biasa aja sih, nggak ada yang spesial.",
    "Pengiriman lambat, barang sampai dalam keadaan rusak. Kecewa parah."
]

predictions = classifier(texts)

for text, pred in zip(texts, predictions):
    print(f"Text: {text}\nPrediction: {pred}\n")

Text: Barangnya bagus banget, sesuai ekspektasi!
Prediction: {'label': 'Communal', 'score': 0.9839069843292236}

Text: Biasa aja sih, nggak ada yang spesial.
Prediction: {'label': 'Communal', 'score': 0.8158960938453674}

Text: Pengiriman lambat, barang sampai dalam keadaan rusak. Kecewa parah.
Prediction: {'label': 'Transactional', 'score': 0.8966231346130371}



In [40]:
import shutil
from IPython.display import FileLink

folder_to_zip = '/kaggle/working/indobert_trained'  
output_filename = '/kaggle/working/indobert_submission' 

print("Zipping the model folder... this might take a minute.")

shutil.make_archive(output_filename, 'zip', folder_to_zip)

print("Done! Click the link below to download:")

FileLink(r'indobert_submission.zip')

Zipping the model folder... this might take a minute.
Done! Click the link below to download:


/kaggle/working/indobert_submission.zip